# Set Up

In [8]:
#load packages
import pandas as pd
import numpy as np
from natsort import natsort_keygen
import mne
from scipy import stats
from nilearn import surface
import os

from utils import *

In [ ]:
# define paths
project_dir = os.chdir('..')
fig_dir = f'{project_dir}figures/'
data_dir = f'{project_dir}data/'
fs_dir = f'{data_dir}freesurfer/'

#define sub list
groups_df = pd.read_csv(f'{data_dir}DFs/groups.csv')
subs = list(groups_df['sub'])

# specify variables
groups = ['dys','typ']
ROIs = ['VWFA1','VWFA2']
hemis = ['LH']


# #subs remove
# ['sub-8','sub-28','sub-31','sub-66','sub-70','sub-82']
# ['sub-1339','sub-617','sub-2127','sub-3215','sub-2628','sub-2734']

# Calculate individual ROI centers
### "centers" are defined as medoids


In [17]:
save_CSV = False
save_labels = False

# set up empty list to collect data
data = []

for hemi in hemis:

    # load fsaverage surface mesh
    mesh = surface.load_surf_mesh(f'{fs_dir}fsaverage/surf/{hemi.lower()}.inflated')
    
    for roi in ROIs:
        for sub in subs:
            # only continue if sub has the label file
            label_path = f'{fs_dir}fsaverage/label/{sub}_T-v-ALL_{roi}_{hemi}.label'
            if os.path.exists(label_path): 
                
                # Load vertices from the label file
                vertices = load_label_file(label_path)

                #make sure the label files contains vertices (sometimes freesurfer can't convert very small labels from fsnative to fsaverage)
                if len(vertices) > 0: 

                     # Get the coordinates of the label's vertices
                    coords = get_surface_coordinates(mesh, vertices)

                    # Compute pairwise distances between the vertices
                    distances = compute_pairwise_distances(coords)

                    # Find the medoid vertex
                    medoid_index = find_medoid(distances)

                    # Get the corresponding vertex ID from the label
                    medoid_vertex = vertices[medoid_index]

                    # Get center coordinages
                    cent_coords = get_surface_coordinates(mesh, medoid_vertex)

                     #gather the data
                    data.append({
                        'sub': sub,
                        'hemi': hemi,
                        'roi': roi, 
                        'X': cent_coords[0], 
                        'Y': cent_coords[1], 
                        'Z': cent_coords[2]
                                })

                    if save_labels:
                        # Save the medoid as a new label
                        output_label_path = f'{data_dir}ROI_centers/{sub}/'
                        if not os.path.exists(output_label_path):
                            os.makedirs(output_label_path)
    
                        label = mne.Label(np.array([medoid_vertex]), hemi=hemi.lower())
    
                        label.save(f'{output_label_path}{sub}_{roi}_medoid')

# Create DataFrame
df = pd.DataFrame(data)   

# save df as csv
if save_CSV:
    
    #merge df with sub group data
    df = pd.merge(df,groups_df,how='left',on='sub')
   
    #merge df with reading score data
    scores = pd.read_csv(f'{data_dir}DFs/wj_scores.csv')
    df = pd.merge(df,scores,how='left',on='sub')
    df.to_csv(f'{data_dir}DFs/coords.csv',index=False)

# Calculate Group ROI centers
## Create group label containing all individual centers

In [49]:
for hemi in hemis:
    for roi in ROIs:
        for group in groups:

            # create empty list to store group medoid vertices
            medoids_vertices = []
            
            # define sub list based on group assignment
            subs = list(groups_df.query('dys_group==@group')['sub'])
            
            
            # loop through subs to calculate individual medoids
            for sub in subs:

                # Define path to the center label
                label_path = f'{data_dir}ROI_centers/{sub}/{sub}_{roi}_medoid-{hemi.lower()}.label'

                if os.path.exists(label_path):  # Only proceed if the sub has a label
                    # Load vertices from the label file
                    vertex = load_label_file(label_path)

                    if len(vertex) > 0: #make sure the label files contains vertices (not sure why some were created without)
                        medoids_vertices.append(vertex)
                        
                        
            # Save the medoid as a new label
            output_label_path = f'{data_dir}ROI_centers/group_centers/'
            if not os.path.exists(output_label_path):
                os.makedirs(output_label_path)
            
            # Flatten medoid arrays
            group_mediods = np.concatenate(medoids_vertices).ravel()
            
            # Remove duplicates by converting to a set
            unique_vertices = set(group_mediods)

            # Sort the unique elements and convert back to a list
            sorted_unique_vertices = sorted(unique_vertices)

            # convert list to label and save
            group_label = mne.Label(np.array(sorted_unique_vertices), hemi=hemi.lower())
            group_label.save(f'{output_label_path}{group}_{roi}s')

            print(f"{group} {roi} {hemi}")                


Saving label to : /oak/stanford/groups/jyeatman/jamie/projects/VWFA_location/data/ROI_centers/group_centers/dys_VWFA1s-lh.label
dys VWFA1 LH
Saving label to : /oak/stanford/groups/jyeatman/jamie/projects/VWFA_location/data/ROI_centers/group_centers/typ_VWFA1s-lh.label
typ VWFA1 LH
Saving label to : /oak/stanford/groups/jyeatman/jamie/projects/VWFA_location/data/ROI_centers/group_centers/dys_VWFA2s-lh.label
dys VWFA2 LH
Saving label to : /oak/stanford/groups/jyeatman/jamie/projects/VWFA_location/data/ROI_centers/group_centers/typ_VWFA2s-lh.label
typ VWFA2 LH


## Calculate group centers

In [ ]:
results = {}  # To store average distances for analysis

# Loop through hemispheres, categories, and sessions
for hemi in hemis:
    for roi in ROIs:
        # Initialize distance lists for this session/category/hemisphere
        group_avg_distances = {group: [] for group in groups}

        for group in groups:
            # Define path to the group label
            group_label_path = f'{data_dir}ROI_centers/group_centers/{group}_{roi}s-{hemi.lower()}.label'

            if os.path.exists(group_label_path):  # Only proceed if the group label exists
                # Load vertices from the group label file
                group_vertices = load_label_file(group_label_path)
                print(len(group_vertices))

                if len(group_vertices) > 0:  # Ensure the label is not empty
                    # load fsaverage surface mesh
                    mesh = surface.load_surf_mesh(f'{fs_dir}fsaverage/surf/{hemi.lower()}.inflated')

                    # Get the coordinates of the group's vertices
                    coords = get_surface_coordinates(mesh, group_vertices)

                    # Compute pairwise distances among the vertices
                    distances = compute_pairwise_distances(coords)

                    # Find the medoid vertex of the group label
                    medoid_index = find_medoid(distances)
#                         print(group_vertices[medoid_index])

                    # Calculate distances from each vertex to the medoid
                    distances_to_medoid = compute_distance_to_medoid(coords, medoid_index)

                    # Append average distance for this group
                    group_avg_distances[group] = list(distances_to_medoid)

                    # Save the label with the group medoid and closest vertices
                    group_label = mne.Label(np.array([group_vertices[medoid_index]]), hemi=hemi.lower())
                    group_label.save(f'{data_dir}ROI_centers/group_centers/{group}_{roi}_medoid')

        

# Calculate Group-Free ROI Centers

In [19]:
save_label = True

results = {}  # To store average distances for analysis

# Loop through hemispheres, categories, and sessions
for hemi in hemis:
    for roi in ROIs:

        # Define path to the group label
        dys_label = load_label_file(f'{data_dir}ROI_centers/group_centers/dys_{roi}_medoid-{hemi.lower()}.label')
        typ_label = load_label_file(f'{data_dir}ROI_centers/group_centers/typ_{roi}_medoid-{hemi.lower()}.label')

        all_vertices = np.concatenate([dys_label,typ_label])
        all_vertices.sort()
        all_vertices=list(set(all_vertices))
                        
        # Get the coordinates of the vertices
        coords = get_surface_coordinates(mesh, all_vertices)

        # Compute pairwise distances among the vertices
        distances = compute_pairwise_distances(coords)

        # Find the medoid vertex of the group label
        medoid_index = find_medoid(distances)

        # Save the label with the group medoid and closest vertices
        if save_label:            
            group_label = mne.Label(np.array([all_vertices[medoid_index]]), hemi=hemi.lower())
            group_label.save(f'{data_dir}ROI_centers/allSubs_{roi}_medoid')



Saving label to : /oak/stanford/groups/jyeatman/jamie/projects/VWFA_location/data/ROI_centers/allSubs_VWFA1_medoid-lh.label
Saving label to : /oak/stanford/groups/jyeatman/jamie/projects/VWFA_location/data/ROI_centers/allSubs_VWFA2_medoid-lh.label
